# DICE — Notebook 1.3 : incertitude et Monte-Carlo — Version étudiante


## 0) Préparation et importations


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
RNG = np.random.default_rng(42)

def clone_params(base, **overrides):
    candidate = Params()
    for name, value in vars(base).items():
        setattr(candidate, name, value)
    for name, value in overrides.items():
        setattr(candidate, name, float(value))
    return candidate

def simulate(base, **overrides):
    candidate = clone_params(base, **overrides)
    path = init_states(candidate)
    path[1:, candidate.i_s] = 0.20
    path[1:, candidate.i_mu] = np.linspace(0.03, 0.60, candidate.nT - 1)
    path = update_path(path, range(1, candidate.nT), candidate)
    return path, candidate

def damage_fraction(path, par):
    lagged_temperature = np.r_[path[0, par.i_T_AT], path[:-1, par.i_T_AT]]
    damages = par.a2 * lagged_temperature ** par.a3
    if par.a4 != 0:
        damages += np.where(lagged_temperature > par.a6,
                            par.a4 * lagged_temperature ** par.a5, 0.0)
    return damages

def quantile_bands(array):
    return np.quantile(np.asarray(array), [0.05, 0.50, 0.95], axis=0)


## Q1) Simuler la référence

Simulez DICE avec l’étalonnage de référence et des contrôles exogènes : taux d’épargne constant $s_t=0{,}20$ et taux de réduction $\mu_t$ passant de 0,03 à 0,60 d’ici 2060. Tracez la température `T_AT` et la production `Y`.


In [ ]:
# Simulez(...) fixe les trajectoires de sauvegarde et de réduction décrites ci-dessus.
p = Params()
baseline, _ = simulate(p)
années = baseline[:, p.i_time]

# Tracer initial[:, p.i_T_AT] et initial[:, p.i_Y] dans deux panneaux.


## Q2) Incertitude sur la sensibilité climatique `T2XCO2`

Nordhaus (2018) retient une distribution lognormale ajustée aux estimations d’Olson et al. (2012). Les paramètres sont $\mu=1{,}107$ et $\sigma=0{,}264$ ; la distribution de référence a une moyenne de 3,13 °C, une médiane de 3,03 °C et un écart-type de 0,843 °C.


### Q2-A) Tirer 1 000 valeurs de `T2XCO2` et tracer l’histogramme

**Indications**
- Utilisez une loi lognormale centrée sur `p.T2XCO2`.
- Définissez `sigma_ecs = math.log(1.25) / norm.ppf(0.95)`.
- Tirez les valeurs avec `np.exp(math.log(p.T2XCO2) + sigma_ecs * RNG.standard_normal(N))`.
- Tracez l’histogramme et affichez les quantiles à 5 %, 50 % et 95 %.


In [ ]:
N = 1000
sigma_ecs = math.log(1.25) / norm.ppf(0.95)
# ecs_draws = ...


### Q2-B) Simuler et stocker chaque trajectoire

**Indications**
- Partez de `p = Params()` et `sim = init_states(p)`.
- Pour chaque tirage, créez un nouvel objet et fixez `p2.T2XCO2 = float(t2x)`.
- Utilisez les mêmes contrôles que dans la référence.
- Exécutez `update_path(sim2, range(1, p2.nT), p2)`.
- Stockez température, production et dommages dans des listes.


In [ ]:
# Votre code pour la question 2-B devrait être ici

### Q2-C) Construire les intervalles de confiance

- Calculez les quantiles avec `np.quantile(arr, [0.05, 0.5, 0.95], axis=0)`.
- Tracez les bandes avec `plt.fill_between` et la médiane avec `plt.plot`.
- Repérez 2100 avec `i2100 = np.argmin(np.abs(années - 2100))` et affichez les trois quantiles.


In [ ]:
# Votre code pour la question 2-C devrait être ici

> Vous avez écrit la réponse ici.

## Q3) Incertitude sur le paramètre de dommages `a2`

Nordhaus (2018) souligne la forte dispersion des estimations de dommages. Nous représentons cette incertitude par une distribution positive dont l’écart-type correspond approximativement à 0,118 % de production par °C².


### Q3-A) Tirer 1 000 valeurs de `a2` et tracer l’histogramme

- Utilisez une loi lognormale positive centrée sur `p.a2`.
- Définissez `sigma_a2 = math.log(2.0) / norm.ppf(0.95)`.
- Tirez les valeurs avec `np.exp(math.log(p.a2) + sigma_a2 * RNG.standard_normal(N))`.
- Affichez les quantiles à 5 %, 50 % et 95 %.


In [ ]:
# Votre code pour la question 3 devrait être ici

### Q3-B) Simuler conjointement sensibilité climatique et dommages

- Parcourez `zip(ecs_draws, a2_draws)`.
- Appelez `simulate(p, T2XCO2=ecs, a2=a2)`.
- Stockez la température, la production et `damage_fraction(path, par)`.
- Convertissez les listes en tableaux NumPy avant de calculer les quantiles.


In [ ]:
# Votre code pour la question 3 devrait être ici

### Q3-C) Tracer les intervalles et interpréter

Calculez les quantiles de température, production et dommages, tracez les bandes de confiance, puis affichez les valeurs obtenues en 2100.


In [ ]:
# Votre code pour la question 3 devrait être ici

> Vous avez écrit la réponse ici.

### Q3-D) Comparer les sources d’incertitude

Superposez les bandes de température à 5–95 % obtenues avec la seule sensibilité climatique puis avec sensibilité et dommages. Comparez en 2100 la largeur `q95 - q05` pour la température, la production et les dommages.


In [ ]:
# Votre code pour la question 3 devrait être ici

> Vous avez écrit la réponse ici.

## Q4) Incertitude sur la décarbonation `deltasig`

Nordhaus (2018) estime une incertitude annuelle sur la tendance de l’intensité carbone. Une régression sur 1960–2015 conduit à une erreur de prévision importante en 2100 ; cette estimation peut encore être trop faible si la série possède une racine unitaire.


### Q4-A) Tirer 1 000 valeurs de `deltasig`

Dans la référence, `deltasig` vaut zéro. Utilisez ici une loi semi-normale positive :
```python
deltasig_draws = np.abs(RNG.normal(loc=0.0, scale=0.02, size=N))
```
Tracez l’histogramme et affichez les quantiles à 5 %, 50 % et 95 %.


In [ ]:
# Votre code pour la question 4 devrait être ici.

### Q4-B) Simuler conjointement les trois incertitudes

- Parcourez `zip(ecs_draws, a2_draws, deltasig_draws)`.
- Appelez `simulate(p, T2XCO2=ecs, a2=a2, deltasig=deltasig)`.
- Stockez température, production et dommages comme précédemment.


In [ ]:
# Votre code pour la question 4 devrait être ici.

### Q4-C) Tracer les intervalles de confiance

Calculez les quantiles à 5 %, 50 % et 95 %, tracez les bandes et affichez les valeurs obtenues en 2100.


In [ ]:
# Votre code pour la question 4 devrait être ici.

In [ ]:
# Intentionally left as a workspace cell.